# Sparkle V2 — Fase 3.2a: Extracción offline de embeddings CNN (rama de apariencia)

**Entorno de ejecución:** Google Colab (GPU T4) o Kaggle Notebooks (GPU) — **requiere el video crudo de DAiSEE** (a diferencia de los notebooks de Fase 3.0/3.1, que solo usan las features ya extraídas).

**Objetivo (Roadmap Fase 3.2, semana 1-2 del plan):** implementar la palanca principal que identifica el diagnóstico del plan (§0, §2): la señal geométrica (landmarks/blendshapes) sola no alcanza ~50% en DAiSEE; los trabajos que superan 65-73% usan apariencia/textura vía CNN sobre el frame o el crop facial. Este notebook agrega ese segundo extractor **sin romper el contrato de privacidad ya vigente**: el crop facial se procesa en memoria, se reduce a un embedding de baja dimensión, y se descarta inmediatamente — exactamente igual a como hoy se descartan los frames tras calcular blendshapes.

**Este notebook corre UNA SOLA VEZ por dataset** (extracción offline, §2.4 del plan): produce embeddings persistidos en `.npz`, igual que ya hacen con las features geométricas. El entrenamiento del modelo temporal (notebook `Fase3_2b`) nunca vuelve a tocar el video.

**Estimación de costo:** 8.416 clips × 16 frames ≈ 135k inferencias de CNN ≈ 1-2 h en GPU T4 (bastante menos que la extracción de landmarks de Fase 2, que ya corrieron sobre los 60 frames completos).

## Cómo correr esto en Colab

1. `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → **T4 GPU**.
2. **Credenciales de Kaggle** (hacen falta para bajar DAiSEE y FER2013). La forma limpia en Colab es el panel de *Secretos* (ícono de llave 🔑 en la barra izquierda): crear `KAGGLE_USERNAME` y `KAGGLE_KEY` con los valores del `kaggle.json` de tu cuenta, y activar el toggle *Acceso al notebook*. Como alternativa, dejar el `kaggle.json` en `MyDrive/SparkleV2/kaggle.json` — la celda lo busca ahí. Como último recurso ofrece subirlo a mano.
3. Correr las celdas en orden. **La celda de extracción (sección 4) se puede cortar y retomar**: lleva su propio registro y saltea los clips ya procesados.

### Diferencias con la versión original del notebook

| Qué | Antes | Ahora |
| :--- | :--- | :--- |
| Descarga de datasets | `!kaggle datasets download` + subida manual de `kaggle.json` | `kagglehub` + Secretos de Colab, con fallback a Drive y a subida manual |
| FER2013 | `ImageDataGenerator` (eliminado en Keras 3) | `keras.utils.image_dataset_from_directory` + `tf.data` |
| Inferencia del extractor | `extractor.predict(...)` por clip | llamada directa `extractor(crops)` — evita el overhead de `predict` en bucle (varias veces más rápido a este tamaño de lote) |
| Lectura de frames | `cap.set(POS_FRAMES)` por frame (seek aleatorio) | decodificación secuencial de una pasada |
| Escritura de los `.npz` | directo a Drive, uno por clip | staging local + sync a Drive cada `SYNC_EVERY` clips (Drive por FUSE es lento con miles de archivos chicos) |
| Reanudación | por split, cambiando una variable a mano | recorre los 3 splits y saltea lo ya hecho, en Drive o en local |

**mediapipe** sí necesita instalarse (Colab no lo trae). Si la instalación pide reiniciar el entorno, reiniciar y volver a correr desde la celda de setup: `PROJECT_DIR` y las credenciales se recalculan solas.

In [ ]:
# mediapipe es lo unico que Colab no trae. TensorFlow/Keras/OpenCV ya estan: NO reinstalar.
%pip install -q mediapipe kagglehub

In [ ]:
# =====================================================================
# SETUP — entorno, Drive, credenciales de Kaggle y verificacion de disco
# =====================================================================
import sys, os, glob, json, math, time, shutil

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/SparkleV2'
    WORK_DIR = '/content'
elif os.path.isdir('/kaggle/input'):
    PROJECT_DIR = '/kaggle/input/sparklev2'; WORK_DIR = '/kaggle/working'
else:
    PROJECT_DIR = os.environ.get('SPARKLE_PROJECT_DIR', './SparkleV2'); WORK_DIR = '.'

assert os.path.isdir(PROJECT_DIR), f'No existe {PROJECT_DIR} (tiene que estar en "Mi unidad")'

LABELS_DIR   = f'{PROJECT_DIR}/labels'
EMB_DIR      = f'{PROJECT_DIR}/embeddings_cnn'        # destino final (Drive)
EMB_LOCAL    = f'{WORK_DIR}/embeddings_cnn'           # staging local (rapido)
BACKBONE_DIR = f'{PROJECT_DIR}/checkpoints_fase3_2'
for split in ('Train', 'Validation', 'Test'):
    os.makedirs(f'{EMB_DIR}/{split}', exist_ok=True)
    os.makedirs(f'{EMB_LOCAL}/{split}', exist_ok=True)
os.makedirs(BACKBONE_DIR, exist_ok=True)
assert os.path.isdir(LABELS_DIR), f'Falta {LABELS_DIR} con TrainLabels.csv / ValidationLabels.csv / TestLabels.csv'

import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
import keras
from keras import layers
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from tqdm.auto import tqdm

gpus = tf.config.list_physical_devices('GPU')
print(f'TF {tf.__version__} | Keras {keras.__version__} | mediapipe {mp.__version__} | GPU: {gpus}')
if not gpus:
    print('\n*** SIN GPU: Entorno de ejecucion -> Cambiar tipo de entorno -> T4 GPU ***\n')

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

CROP_SIZE    = 112       # MobileNetV2 alpha=0.35 @112 (S2.2 del plan)
N_EMB_FRAMES = 16        # muestreo temporal disperso (S2.3): 0.8-1.6 fps sobre 10s
EMB_DIM_PROJ = 128       # proyeccion final del embedding (64-128d, S2.1)
ALPHA        = 0.35
SYNC_EVERY   = 200       # cada cuantos clips se sincroniza el staging local con Drive

# --- Credenciales de Kaggle: Secretos de Colab -> kaggle.json en Drive -> subida manual ---
def setup_kaggle_credentials():
    if os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'):
        return 'ya configuradas'
    if IN_COLAB:
        try:
            from google.colab import userdata
            os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
            os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
            return 'Secretos de Colab'
        except Exception:
            pass
    for cand in (f'{PROJECT_DIR}/kaggle.json', '/content/drive/MyDrive/kaggle.json',
                 os.path.expanduser('~/.kaggle/kaggle.json')):
        if os.path.exists(cand):
            with open(cand) as f:
                c = json.load(f)
            os.environ['KAGGLE_USERNAME'] = c['username']; os.environ['KAGGLE_KEY'] = c['key']
            return f'kaggle.json en {cand}'
    if IN_COLAB:
        from google.colab import files
        print('No encontre credenciales. Subi tu kaggle.json:')
        up = files.upload()
        name = list(up)[0]
        c = json.loads(up[name])
        os.environ['KAGGLE_USERNAME'] = c['username']; os.environ['KAGGLE_KEY'] = c['key']
        return 'subida manual'
    raise RuntimeError('Sin credenciales de Kaggle')

print('Credenciales de Kaggle:', setup_kaggle_credentials())

total, used, free = shutil.disk_usage(WORK_DIR)
print(f'Disco en {WORK_DIR}: {free/1e9:.0f} GB libres de {total/1e9:.0f} GB '
      f'(DAiSEE crudo necesita ~25-30 GB entre el zip y el descomprimido)')

## 0.1 Descarga de DAiSEE (video crudo) e índice de videos

`kagglehub` descarga a la caché local (`~/.cache/kagglehub`) y devuelve la ruta; si el dataset ya está en el runtime, no vuelve a bajarlo. La descarga es de decenas de GB: si Colab corta la sesión, esta celda hay que volver a correrla desde cero (la caché no sobrevive al reinicio del runtime).

In [ ]:
import kagglehub

DAISEE_SLUG = 'olgaparfenova/daisee'

def find_dataset_root(base):
    '''DAiSEE viene como .../DAiSEE/DataSet/{Train,Validation,Test}. Segun el mirror,
    kagglehub puede devolver un nivel mas arriba o mas abajo: se busca el que tenga
    las tres carpetas de split.'''
    for root, dirs, _ in os.walk(base):
        if {'Train', 'Validation', 'Test'}.issubset(set(dirs)):
            return root
    return base

t0 = time.time()
daisee_path = kagglehub.dataset_download(DAISEE_SLUG)
RAW_DATASET_DIR = find_dataset_root(daisee_path)
print(f'DAiSEE en {RAW_DATASET_DIR}  ({(time.time()-t0)/60:.1f} min)')

def load_labels(split):
    df = pd.read_csv(f'{LABELS_DIR}/{split}Labels.csv')
    df.columns = [c.strip() for c in df.columns]
    return df

labels = {s: load_labels(s) for s in ('Train', 'Validation', 'Test')}
for s, df in labels.items():
    print(f'{s}Labels.csv: {len(df)} filas')

video_index = {}
for split in ('Train', 'Validation', 'Test'):
    for ext in ('avi', 'mp4'):
        for p in glob.glob(f'{RAW_DATASET_DIR}/{split}/**/*.{ext}', recursive=True):
            video_index[os.path.basename(p)] = p
print(f'Videos indexados: {len(video_index)}')
assert len(video_index) > 0, 'No se indexo ningun video: revisar RAW_DATASET_DIR'

## 1. FaceLandmarker (mismo modelo que Fase 2/3) para obtener el bounding box del rostro

Se reutiliza el `FaceLandmarker` de MediaPipe ya validado en la extracción de blendshapes — evita introducir un segundo detector facial. El bounding box se deriva de los landmarks (min/max x,y con margen), no hace falta un detector de caras separado.

In [ ]:
import urllib.request

MODEL_URL  = ('https://storage.googleapis.com/mediapipe-models/face_landmarker/'
              'face_landmarker/float16/latest/face_landmarker.task')
MODEL_PATH = f'{WORK_DIR}/face_landmarker.task'
if not os.path.exists(MODEL_PATH):
    print('Descargando face_landmarker.task ...')
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
print(f'FaceLandmarker: {os.path.getsize(MODEL_PATH)/1e6:.1f} MB')

def create_face_landmarker():
    base_options = mp_python.BaseOptions(model_asset_path=MODEL_PATH)
    options = mp_vision.FaceLandmarkerOptions(
        base_options=base_options, running_mode=mp_vision.RunningMode.IMAGE,
        num_faces=1, min_face_detection_confidence=0.5)
    return mp_vision.FaceLandmarker.create_from_options(options)

def crop_face_from_landmarks(frame_bgr, landmarks, crop_size=CROP_SIZE, margin=0.25):
    '''Bounding box a partir de min/max de todos los landmarks + margen relativo,
    recorte cuadrado (evita distorsionar el rostro) y resize a crop_size x crop_size.
    Devuelve RGB uint8; preprocess_input se aplica aparte.'''
    h, w = frame_bgr.shape[:2]
    xs = np.array([lm.x for lm in landmarks]) * w
    ys = np.array([lm.y for lm in landmarks]) * h
    x_min, x_max, y_min, y_max = xs.min(), xs.max(), ys.min(), ys.max()
    bw, bh = x_max - x_min, y_max - y_min
    side = max(bw, bh) * (1 + margin)
    cx, cy = (x_min + x_max) / 2, (y_min + y_max) / 2
    x0, y0 = int(max(0, cx - side / 2)), int(max(0, cy - side / 2))
    x1, y1 = int(min(w, cx + side / 2)), int(min(h, cy + side / 2))
    if x1 <= x0 or y1 <= y0:
        return None
    crop = cv2.resize(frame_bgr[y0:y1, x0:x1], (crop_size, crop_size), interpolation=cv2.INTER_AREA)
    return cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)

def extract_crop_from_frame(detector, frame_bgr):
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
    result = detector.detect(mp_image)
    if not result.face_landmarks:
        return None
    return crop_face_from_landmarks(frame_bgr, result.face_landmarks[0])

## 2. Backbone CNN — MobileNetV2 α=0,35 @112 (fine-tuned en FER2013 o ImageNet directo)

**Opción A (recomendada, `USE_FER_FINETUNED = True`):** fine-tuning corto de MobileNetV2 en FER2013 (7 clases de emoción, imágenes 48×48 en escala de grises — se convierten a RGB 112×112 con resize). Al terminar, se congela el backbone y se usa la penúltima capa (antes del clasificador de 7 clases) como extractor de "textura afectiva".

**Opción B (`USE_FER_FINETUNED = False`):** MobileNetV2 con pesos ImageNet directos, sin fine-tuning. Más rápido de poner en marcha pero el plan advierte que rinde peor — usar solo para validar el pipeline end-to-end antes de invertir en el fine-tuning.

In [ ]:
USE_FER_FINETUNED = True     # False = Opcion B (ImageNet directo, rinde peor; solo para smoke test)
FER_SLUG   = 'msambare/fer2013'
FER_EPOCHS = 10

def build_backbone(alpha=ALPHA, crop_size=CROP_SIZE, weights='imagenet'):
    return keras.applications.MobileNetV2(
        input_shape=(crop_size, crop_size, 3), alpha=alpha,
        include_top=False, weights=weights, pooling='avg')

BACKBONE_CKPT = f'{BACKBONE_DIR}/mobilenetv2_fer_finetuned.keras'

if USE_FER_FINETUNED and os.path.exists(BACKBONE_CKPT):
    backbone = keras.models.load_model(BACKBONE_CKPT)
    backbone.trainable = False
    print('Backbone fine-tuned recuperado de Drive (no se reentrena).')

elif USE_FER_FINETUNED:
    fer_root = kagglehub.dataset_download(FER_SLUG)
    fer_train_dir = next(p for p in glob.glob(f'{fer_root}/**/train', recursive=True) if os.path.isdir(p))
    print('FER2013 train dir:', fer_train_dir)

    # Keras 3: ImageDataGenerator ya no existe -> image_dataset_from_directory + tf.data
    train_ds = keras.utils.image_dataset_from_directory(
        fer_train_dir, labels='inferred', label_mode='categorical',
        color_mode='rgb', image_size=(CROP_SIZE, CROP_SIZE), batch_size=64,
        validation_split=0.1, subset='training', seed=SEED, shuffle=True)
    val_ds = keras.utils.image_dataset_from_directory(
        fer_train_dir, labels='inferred', label_mode='categorical',
        color_mode='rgb', image_size=(CROP_SIZE, CROP_SIZE), batch_size=64,
        validation_split=0.1, subset='validation', seed=SEED, shuffle=False)
    n_fer_classes = len(train_ds.class_names)
    print(f'FER2013: {n_fer_classes} clases {train_ds.class_names}')

    prep = keras.applications.mobilenet_v2.preprocess_input
    train_ds = train_ds.map(lambda x, y: (prep(x), y), num_parallel_calls=tf.data.AUTOTUNE
                            ).prefetch(tf.data.AUTOTUNE)
    val_ds   = val_ds.map(lambda x, y: (prep(x), y), num_parallel_calls=tf.data.AUTOTUNE
                          ).prefetch(tf.data.AUTOTUNE)

    backbone = build_backbone(weights='imagenet')
    backbone.trainable = True          # fine-tuning completo (el backbone es chico, alpha=0.35)
    fer_head = layers.Dense(n_fer_classes, activation='softmax', name='fer_head')(backbone.output)
    fer_model = keras.Model(backbone.input, fer_head)
    fer_model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4),
                      loss='categorical_crossentropy', metrics=['accuracy'])
    fer_model.fit(train_ds, validation_data=val_ds, epochs=FER_EPOCHS,
                  callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True,
                                                           monitor='val_accuracy', mode='max')])
    backbone.trainable = False
    backbone.save(BACKBONE_CKPT)
    print('Backbone fine-tuned en FER2013, congelado y guardado en Drive.')

else:
    backbone = build_backbone(weights='imagenet')
    backbone.trainable = False
    print('Backbone con pesos ImageNet directos (Opcion B, rendimiento esperado menor).')

# Extractor final: backbone congelado + proyeccion a EMB_DIM_PROJ
inp_crop = keras.Input(shape=(CROP_SIZE, CROP_SIZE, 3), name='crop')
feat = backbone(inp_crop, training=False)
emb_out = layers.Dense(EMB_DIM_PROJ, activation=None, name='embedding_projection')(feat)
extractor_model = keras.Model(inp_crop, emb_out, name='cnn_extractor')
extractor_model.trainable = False
print(f'Extractor listo: {extractor_model.count_params():,} parametros, '
      f'salida {extractor_model.output_shape}')

# La proyeccion de 128-d se inicializa al azar y queda FIJA para los 3 splits: define un
# subespacio aleatorio estable del embedding del backbone (Johnson-Lindenstrauss). Se
# guarda para poder reproducir la extraccion o reanudarla en otra sesion de Colab.
extractor_model.save(f'{BACKBONE_DIR}/cnn_extractor.keras')
print('Extractor guardado en', f'{BACKBONE_DIR}/cnn_extractor.keras')

## 3. Función de procesamiento por clip — muestreo disperso (16 frames) + embedding

A diferencia de la extracción geométrica (60 frames/clip), acá se muestrean solo 16 frames por clip (§2.3 del plan: 0,8-1,6 fps es suficiente porque la textura facial cambia lento). El frame se descarta inmediatamente después de convertirlo en embedding — nunca se persiste el crop ni el frame.

**Cambio de rendimiento respecto de la versión original:** los frames se leen **secuencialmente** (una sola pasada del decodificador) en lugar de hacer `cap.set(CAP_PROP_POS_FRAMES)` por cada uno de los 16 frames — el seek aleatorio en `.avi` obliga al decodificador a rebobinar hasta el keyframe anterior cada vez. Y el embedding se calcula con `extractor(crops)` en vez de `extractor.predict(crops)`, que en un bucle de miles de llamadas agrega un overhead considerable por invocación.

In [ ]:
def sample_frames_sequential(video_path, n_frames=N_EMB_FRAMES):
    '''Devuelve los n_frames muestreados uniformemente, leyendo el video de una
    sola pasada (mucho mas rapido que hacer seek por frame en .avi).'''
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release(); return None
    wanted = np.linspace(0, total - 1, n_frames).astype(int)
    wanted_set = {}
    for pos, idx in enumerate(wanted):
        wanted_set.setdefault(int(idx), []).append(pos)

    frames = [None] * n_frames
    i, max_idx = 0, int(wanted.max())
    while i <= max_idx:
        ret, frame = cap.read()
        if not ret:
            break
        if i in wanted_set:
            for pos in wanted_set[i]:
                frames[pos] = frame
        i += 1
    cap.release()
    return frames

def process_clip_embeddings(video_path, detector, extractor, n_frames=N_EMB_FRAMES):
    frames = sample_frames_sequential(video_path, n_frames)
    if frames is None:
        return None
    crops = np.zeros((n_frames, CROP_SIZE, CROP_SIZE, 3), dtype=np.float32)
    valid_mask = np.zeros(n_frames, dtype=bool)
    for pos, frame in enumerate(frames):
        if frame is None:
            continue
        crop = extract_crop_from_frame(detector, frame)   # el frame se descarta al salir del scope
        if crop is not None:
            crops[pos] = keras.applications.mobilenet_v2.preprocess_input(crop.astype(np.float32))
            valid_mask[pos] = True
    if not valid_mask.any():
        return np.full((n_frames, EMB_DIM_PROJ), np.nan, dtype=np.float32)
    embeddings = np.asarray(extractor(crops, training=False))   # (16, EMB_DIM_PROJ)
    embeddings = embeddings.astype(np.float32)
    embeddings[~valid_mask] = np.nan   # frames sin rostro -> NaN (mismo criterio que las geometricas)
    return embeddings

# --- Prueba en un subset chico antes de lanzar el batch completo ---
test_clips = labels['Train'].head(10)
detector = create_face_landmarker()
elapsed, n_ok = 0.0, 0
try:
    t0 = time.time()
    for _, row in test_clips.iterrows():
        clip_id = row['ClipID']
        if clip_id not in video_index:
            print(f'{clip_id}: sin video fisico'); continue
        emb = process_clip_embeddings(video_index[clip_id], detector, extractor_model)
        pct_nan = np.isnan(emb).any(axis=1).mean() * 100
        print(f'{clip_id}: {pct_nan:5.1f}% frames sin rostro | shape {emb.shape} '
              f'| norma media {np.nanmean(np.linalg.norm(np.nan_to_num(emb), axis=1)):.3f}')
        n_ok += 1
    elapsed = time.time() - t0
finally:
    detector.close()

if n_ok:
    print(f'\nTiempo: {elapsed:.1f}s para {n_ok} clips ({elapsed/n_ok:.2f}s/clip)')
    n_total = sum(len(df) for df in labels.values())
    print(f'Estimado dataset completo ({n_total} clips): {elapsed/n_ok*n_total/60:.0f} minutos')

**Antes de lanzar el batch completo:** confirmar que el % de frames sin rostro es razonable (similar al de la extracción geométrica) y que la norma de los embeddings no es 0 ni constante entre clips distintos.

## 4. Extracción completa — los 3 splits, reanudable

Esta celda es **idempotente**: se puede cortar (desconexión de Colab, límite de sesión, `Ctrl+M I`) y volver a correr; saltea todo lo que ya esté hecho.

Cómo funciona la reanudación:

- Los `.npz` se escriben primero en `EMB_LOCAL` (disco del runtime, rápido) y se sincronizan a Drive cada `SYNC_EVERY` clips y al terminar cada split.
- Al arrancar, considera "ya hecho" cualquier clip que exista **en Drive o en local**, así que si el runtime se reinicia solo recupera lo que llegó a sincronizar.
- El registro por clip (`{split}_processing_log.csv`) también se persiste en Drive.

Si Colab corta la sesión, hay que volver a bajar DAiSEE (la caché no sobrevive al reinicio del runtime), pero **los embeddings ya extraídos no se pierden**.

In [ ]:
SPLITS = ['Train', 'Validation', 'Test']   # dejar solo uno para correr por partes

def sync_to_drive(split):
    src, dst = f'{EMB_LOCAL}/{split}', f'{EMB_DIR}/{split}'
    n = 0
    for f in os.listdir(src):
        d = f'{dst}/{f}'
        if not os.path.exists(d):
            shutil.copy2(f'{src}/{f}', d); n += 1
    return n

def already_done_ids(split):
    done = set()
    for base in (f'{EMB_DIR}/{split}', f'{EMB_LOCAL}/{split}'):
        if os.path.isdir(base):
            done |= {os.path.splitext(f)[0] for f in os.listdir(base) if f.endswith('.npz')}
    return done

for SPLIT in SPLITS:
    df_split = labels[SPLIT]
    out_dir  = f'{EMB_LOCAL}/{SPLIT}'
    log_path = f'{EMB_DIR}/{SPLIT}_processing_log.csv'
    log_rows = pd.read_csv(log_path).to_dict('records') if os.path.exists(log_path) else []
    done = already_done_ids(SPLIT)
    pendientes = [r for _, r in df_split.iterrows()
                  if os.path.splitext(r['ClipID'])[0] not in done]
    print(f'\n=== {SPLIT}: {len(done)} ya procesados, {len(pendientes)} pendientes '
          f'de {len(df_split)} ===')
    if not pendientes:
        sync_to_drive(SPLIT); continue

    detector = create_face_landmarker()
    procesados = 0
    try:
        for row in tqdm(pendientes, desc=SPLIT):
            clip_id = row['ClipID']
            clip_key = os.path.splitext(clip_id)[0]
            if clip_id not in video_index:
                log_rows.append({'ClipID': clip_id, 'status': 'sin_video_fisico'}); continue
            try:
                emb = process_clip_embeddings(video_index[clip_id], detector, extractor_model)
                if emb is None:
                    log_rows.append({'ClipID': clip_id, 'status': 'video_ilegible'}); continue
                pct_nan = float(np.isnan(emb).any(axis=1).mean() * 100)
                np.savez_compressed(f'{out_dir}/{clip_key}.npz', embeddings=emb.astype(np.float32))
                log_rows.append({'ClipID': clip_id, 'status': 'ok', 'pct_nan': pct_nan})
                procesados += 1
            except Exception as e:
                log_rows.append({'ClipID': clip_id, 'status': f'error: {type(e).__name__}: {e}'})
            if procesados and procesados % SYNC_EVERY == 0:
                n = sync_to_drive(SPLIT)
                pd.DataFrame(log_rows).to_csv(log_path, index=False)
                tqdm.write(f'  [checkpoint] {procesados} clips | {n} archivos nuevos a Drive')
    finally:
        detector.close()
        n = sync_to_drive(SPLIT)
        pd.DataFrame(log_rows).to_csv(log_path, index=False)
        print(f'{SPLIT}: sincronizados {n} archivos a Drive. Log en {log_path}')

# Resumen final
for SPLIT in SPLITS:
    n_drive = len(glob.glob(f'{EMB_DIR}/{SPLIT}/*.npz'))
    log_path = f'{EMB_DIR}/{SPLIT}_processing_log.csv'
    if os.path.exists(log_path):
        st = pd.read_csv(log_path)['status'].astype(str)
        st = st.where(st.str.startswith('error'), st).str.split(':').str[0]
        print(f'{SPLIT}: {n_drive} .npz en Drive | estados: {dict(st.value_counts())}')
    else:
        print(f'{SPLIT}: {n_drive} .npz en Drive')

## 5. Control de calidad de los embeddings

Antes de pasar a 3.2b conviene confirmar que los embeddings tienen señal: si todos los clips dan casi el mismo vector, algo está mal en el recorte o en el backbone y la fusión no va a poder aportar nada.

In [ ]:
qc_files = sorted(glob.glob(f'{EMB_DIR}/Train/*.npz'))[:300]
if qc_files:
    M = np.stack([np.load(f)['embeddings'] for f in qc_files])       # (n, 16, 128)
    clip_mean = np.nanmean(M, axis=1)                                 # (n, 128)
    print(f'{len(qc_files)} clips de QC | shape {M.shape}')
    print(f'% de frames sin rostro: {np.isnan(M).any(axis=2).mean()*100:.2f}%')
    print(f'Norma media por clip: {np.nanmean(np.linalg.norm(np.nan_to_num(clip_mean), axis=1)):.3f}')
    print(f'Desvio ENTRE clips (media por dimension): {np.nanstd(clip_mean, axis=0).mean():.4f}')
    print(f'Desvio DENTRO de un clip (entre sus 16 frames): {np.nanmean(np.nanstd(M, axis=1)):.4f}')
    cm = np.nan_to_num(clip_mean)
    cmn = cm / (np.linalg.norm(cm, axis=1, keepdims=True) + 1e-8)
    sim = cmn @ cmn.T
    iu = np.triu_indices(len(cmn), k=1)
    print(f'Similitud coseno media entre clips distintos: {sim[iu].mean():.3f} '
          f'({"OK" if sim[iu].mean() < 0.98 else "SOSPECHOSO: embeddings casi identicos"})')
else:
    print('Todavia no hay embeddings en Drive: correr la celda de extraccion.')

## 6. Export del extractor CNN a TFLite

El criterio de salida de Fase 3.2 mira el tamaño **total** en el dispositivo (extractor + modelo de fusión). Esta celda convierte el extractor acá, donde ya está entrenado, para que el notebook 3.2b solo tenga que sumar el número.

Mismo camino que en 3.1/3.2b: `ExportArchive` con batch fijo → SavedModel → `TFLiteConverter.from_saved_model`, porque en Keras 3 `from_keras_model` ya no acepta modelos Keras directamente.

In [ ]:
import tempfile

def keras_to_tflite(model, input_specs, representative_dataset=None):
    sm_dir = tempfile.mkdtemp(prefix='sparkle_sm_')
    shutil.rmtree(sm_dir, ignore_errors=True)
    exported = False
    try:
        from keras.export import ExportArchive
        archive = ExportArchive(); archive.track(model)
        fn = (lambda x: model(x, training=False)) if len(input_specs) == 1 \
             else (lambda *xs: model(list(xs), training=False))
        archive.add_endpoint(name='serve', fn=fn, input_signature=list(input_specs))
        archive.write_out(sm_dir); exported = True
    except Exception as e:
        print('  ExportArchive fallo:', repr(e)[:200])
        try:
            model.export(sm_dir); exported = True
        except Exception as e2:
            print('  model.export() fallo:', repr(e2)[:200])
    converter = (tf.lite.TFLiteConverter.from_saved_model(sm_dir) if exported
                 else tf.lite.TFLiteConverter.from_keras_model(model))
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    if representative_dataset is not None:
        converter.representative_dataset = representative_dataset
    out = converter.convert()
    shutil.rmtree(sm_dir, ignore_errors=True)
    return out

specs = [tf.TensorSpec((1, CROP_SIZE, CROP_SIZE, 3), tf.float32, name='crop')]
tfl_float = keras_to_tflite(extractor_model, specs)
print(f'Extractor CNN TFLite float: {len(tfl_float)/1024:.1f} KB')

# Dataset representativo: crops reales de unos pocos clips de Train
rep_crops = []
det = create_face_landmarker()
try:
    for _, row in labels['Train'].head(12).iterrows():
        if row['ClipID'] not in video_index:
            continue
        for fr in (sample_frames_sequential(video_index[row['ClipID']], 8) or []):
            if fr is None:
                continue
            c = extract_crop_from_frame(det, fr)
            if c is not None:
                rep_crops.append(keras.applications.mobilenet_v2.preprocess_input(c.astype(np.float32)))
        if len(rep_crops) >= 100:
            break
finally:
    det.close()
print(f'Crops representativos para la cuantizacion: {len(rep_crops)}')

def representative_dataset():
    for c in rep_crops:
        yield [c[None, ...].astype('float32')]

tfl_int8 = keras_to_tflite(extractor_model, specs,
                           representative_dataset if rep_crops else None)
print(f'Extractor CNN TFLite int8: {len(tfl_int8)/1024:.1f} KB  '
      f'(estimacion del plan para este backbone: ~1,7 MB)')

with open(f'{BACKBONE_DIR}/cnn_extractor.tflite', 'wb') as f:
    f.write(tfl_int8)
with open(f'{BACKBONE_DIR}/cnn_extractor_float.tflite', 'wb') as f:
    f.write(tfl_float)
print('Guardado en', f'{BACKBONE_DIR}/cnn_extractor.tflite')

## Checkpoint de la Fase 3.2a

1. Correr la sección 3 (subset de 10 clips) y confirmar que los embeddings tienen norma no-trivial y variabilidad entre clips (si todos dan casi el mismo vector, algo está mal en el crop o el backbone).
2. Repetir la celda de batch completo para `Train`, `Validation` y `Test`.
3. **No se persiste ningún crop ni frame** — solo el vector de embedding de 128-d por frame muestreado, igual contrato de privacidad que ya usan (§2.5 del plan: nunca sale del dispositivo, se descarta tras la inferencia; acá aplica lo mismo a nivel de extracción offline del dataset de entrenamiento).
4. Con los tres splits listos, seguir con `Sparkle_V2_Fase3_2b_Fusion_Apariencia_Geometria.ipynb` para entrenar el modelo de fusión.